[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-09b-benchmarks-elo-and-contamination.ipynb)

# Advanced Discussion 9 (continued) — reading benchmarks and arena rankings critically

Benchmark margins of error, paired comparisons, option-position bias, online Elo versus Bradley-Terry, bootstrapped ranks, and contamination.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in about half a minute.

In [ ]:
%pip install -q numpy

In [ ]:
import numpy as np, math
rng = np.random.RandomState(0)

## 1. a multiple-choice benchmark score is a noisy estimate

In [ ]:
print("accuracy of a model that is truly 70% correct on a 4-option benchmark; 95% interval half-width from the binomial")
for n in (50, 100, 500, 1000, 5000, 14000):
    print("  %5d questions: +/- %.1f points" % (n, 100 * 1.96 * math.sqrt(0.7 * 0.3 / n)))
print("random guessing on 4 options scores 25%; a model at 40% is 15 points above chance, not 40 points above zero")
# two models 2 points apart on 1000 questions, evaluated on the same questions (paired)
def separation(n, pa, pb, rho=0.6, sims=2000, seed=1):
    r = np.random.RandomState(seed); hits = 0
    for _ in range(sims):
        u = r.rand(n); v = np.where(r.rand(n) < rho, u, r.rand(n)); d = (v < pb).astype(float) - (u < pa).astype(float)
        se = d.std(ddof=1) / math.sqrt(n); hits += d.mean() - 1.96 * se > 0
    return hits / sims
print("\nchance a paired comparison shows the better model is better (true gap, questions):")
for gap in (0.02, 0.05):
    print("  gap %.0f points: " % (100 * gap) + "  ".join("n=%d: %.0f%%" % (n, 100 * separation(n, 0.60, 0.60 + gap)) for n in (100, 500, 1000, 5000)))

## 2. answer-position sensitivity: does the score depend on where the correct option sits?

In [ ]:
def mc_accuracy(bias_to_A, n=4000, skill=0.55, seed=0):
    r = np.random.RandomState(seed); acc = {p: [0, 0] for p in range(4)}
    for _ in range(n):
        pos = r.randint(4); knows = r.rand() < skill
        if knows: pick = pos if r.rand() > bias_to_A * (pos != 0) else 0          # sometimes drifts to option A even when it knows
        else: pick = 0 if r.rand() < bias_to_A else r.randint(4)                   # guesses lean toward A
        acc[pos][0] += pick == pos; acc[pos][1] += 1
    return {p: v[0] / v[1] for p, v in acc.items()}
res = mc_accuracy(0.2)
print("\nmodel with a mild lean toward option A: accuracy by position of the correct answer")
print("  " + "  ".join("%s: %.3f" % ("ABCD"[p], a) for p, a in res.items()) + "   (an unbiased model would be flat)")
r0 = mc_accuracy(0.0)
print("  unbiased:                         " + "  ".join("%s: %.3f" % ("ABCD"[p], a) for p, a in r0.items()))
print("  overall score with the correct option placed at random: biased %.3f | unbiased %.3f" % (np.mean(list(res.values())), np.mean(list(r0.values()))))
print("  if a benchmark happened to put the answer at A in 40%% of items: biased model scores %.3f, unbiased %.3f" % (0.4 * res[0] + 0.2 * (res[1] + res[2] + res[3]), 0.4 * r0[0] + 0.2 * (r0[1] + r0[2] + r0[3])))

## 3. arena-style ranking: pairwise votes -> Elo (online) vs Bradley-Terry (fit to all votes)

In [ ]:
names = ["A", "B", "C", "D", "E", "F"]; true = np.array([1.2, 1.0, 0.7, 0.4, 0.2, 0.0])
def make_votes(n, seed):
    r = np.random.RandomState(seed); v = []
    for _ in range(n):
        i, j = r.choice(6, 2, replace=False); p = 1 / (1 + math.exp(-(true[i] - true[j]))); v.append((i, j, r.rand() < p))       # True = i wins
    return v
def elo(votes, k=32):
    R = np.full(6, 1000.0)
    for i, j, iw in votes:
        e = 1 / (1 + 10 ** ((R[j] - R[i]) / 400)); s = 1.0 if iw else 0.0; R[i] += k * (s - e); R[j] -= k * (s - e)
    return R
def bt(votes, iters=200):
    n = 6; W = np.zeros((n, n))
    for i, j, iw in votes:
        if iw: W[i, j] += 1
        else: W[j, i] += 1
    p = np.ones(n)
    for _ in range(iters):
        for i in range(n):
            num = W[i].sum(); den = sum((W[i, j] + W[j, i]) / (p[i] + p[j]) for j in range(n) if j != i); p[i] = num / den
        p /= np.exp(np.log(p).mean())
    return 400 * np.log10(p) + 1000
V = make_votes(3000, 0)
order = np.argsort(-(-true)); print("\ntrue strengths (log-odds):", dict(zip(names, true.tolist())))
r_elo = elo(V); r_bt = bt(V)
print("after 3000 votes: Elo (k=32) ", {k: int(round(v)) for k, v in zip(names, r_elo)})
print("                  Bradley-Terry", {k: int(round(v)) for k, v in zip(names, r_bt)})
# order sensitivity of online Elo: shuffle the same votes
shuffled = []
for s in range(200):
    idx = np.random.RandomState(s).permutation(len(V)); shuffled.append(elo([V[i] for i in idx]))
shuffled = np.array(shuffled); print("same 3000 votes, 200 random orders: Elo rating of A ranges over %.0f-%.0f (sd %.1f); B-T is order-independent" % (shuffled[:, 0].min(), shuffled[:, 0].max(), shuffled[:, 0].std()))
rank_flip = np.mean([np.argsort(-x)[0] != 0 for x in shuffled]); print("share of orders in which the top-ranked model is not A: %.0f%%" % (100 * rank_flip))

## 4. uncertainty and votes needed: bootstrap the ratings; how often is the ranking right?

In [ ]:
def bootstrap_rank(votes, B=200, seed=5):
    r = np.random.RandomState(seed); ranks = []
    for _ in range(B):
        idx = r.randint(0, len(votes), len(votes)); ranks.append(np.argsort(np.argsort(-bt([votes[i] for i in idx]))) + 1)
    ranks = np.array(ranks); return ranks
print("\nbootstrap 95% interval of each model's rank, by number of votes")
for n in (300, 1000, 3000):
    ranks = bootstrap_rank(make_votes(n, 3), B=100)
    print("  %4d votes: " % n + "  ".join("%s %d-%d" % (nm, np.percentile(ranks[:, i], 2.5), np.percentile(ranks[:, i], 97.5)) for i, nm in enumerate(names)))
def p_correct_order(n, i, j, trials=100):
    ok = 0
    for t in range(trials):
        b = bt(make_votes(n, 100 + t)); ok += b[i] > b[j]
    return ok / trials
print("\nchance the fitted ratings put the truly stronger model ahead (A vs B differ by 0.2 log-odds; A vs D by 0.8):")
for n in (300, 1000, 3000):
    print("  %4d votes: A over B %.0f%% | A over D %.0f%%" % (n, 100 * p_correct_order(n, 0, 1), 100 * p_correct_order(n, 0, 3)))

## 5. a benchmark score after contamination

In [ ]:
print("\nbenchmark with 1000 questions; a model truly answers 55% correctly, and has memorised a fraction c of the questions")
for c in (0.0, 0.02, 0.05, 0.10, 0.20):
    print("  contaminated share %4.0f%%: reported score %.1f%%" % (100 * c, 100 * (c * 1.0 + (1 - c) * 0.55)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.